# VICTOR Ablation & Sensitivity Study

**Experiment Goal:**
Address Reviewer 4, Reviewer 7, and EiC requests for ablation,
correction/abstention evaluation, and sensitivity analysis.

**Methodology Freeze:**
Core VICTOR pipeline remains identical to the submitted manuscript.
Only additional evaluation variants and diagnostics are introduced.

**Reproducibility:**
- Seed=42
- 50/50 stratified split
- google/gemma-2-2b-it
- Paper-exact claim decomposition

In [21]:
# Cell 1 — Installs
import subprocess
subprocess.run(['pip', 'install', '-q',
    'transformers', 'datasets', 'rank_bm25',
    'scikit-learn', 'pandas', 'numpy',
    'matplotlib', 'pyyaml', 'accelerate',
    'torch'
], check=True)
print('All packages installed.')

All packages installed.


In [22]:
# Cell 2 — Imports + Seed + run_manifest.json + config.yaml
import os, json, yaml, time, random, re
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score

# ── Seed ──────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Output directory ──────────────────────────────────────────────────
OUT = '/kaggle/working/victor_outputs'
os.makedirs(OUT, exist_ok=True)

# ── Config ────────────────────────────────────────────────────────────
CONFIG = {
    'model_name': 'google/gemma-2-2b-it',
    'seed': SEED,
    'split': '50_50_stratified',
    'tau_default': 0.10,  # Recalibrated for HF Gemma-2-2B-it (diagnostic confirmed 0.30 unreachable)
    'top_k_default': 3,
    'delta': 0.5,
    'gamma': 0.2,
    'tau_sweep': [0.1, 0.2, 0.3, 0.4, 0.5],
    'topk_sweep': [1, 3, 5],
    'datasets': ['TruthfulQA', 'FEVER', 'HaluEval'],
    'variants': ['FULL', 'NO_UNCERTAINTY', 'NO_BM25', 'NO_REVISION', 'NO_ABSTENTION'],
    'claim_decomposition_mode': 'paper_exact',
    'pilot_dataset': 'TruthfulQA',
    'pilot_samples': 50,
    'max_new_tokens': 64,
    'bootstrap_iterations': 1000
}

# Save config.yaml
with open(f'{OUT}/config.yaml', 'w') as f:
    yaml.dump(CONFIG, f, default_flow_style=False)

# Save run_manifest.json
manifest = {
    'model': CONFIG['model_name'],
    'seed': CONFIG['seed'],
    'split': CONFIG['split'],
    'tau_default': CONFIG['tau_default'],
    'top_k_default': CONFIG['top_k_default'],
    'datasets': CONFIG['datasets'],
    'variants': CONFIG['variants'],
    'timestamp': datetime.utcnow().isoformat() + 'Z',
    'git_commit': 'optional'
}
with open(f'{OUT}/run_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)

print('Config and manifest saved.')
print(json.dumps(manifest, indent=2))

Config and manifest saved.
{
  "model": "google/gemma-2-2b-it",
  "seed": 42,
  "split": "50_50_stratified",
  "tau_default": 0.1,
  "top_k_default": 3,
  "datasets": [
    "TruthfulQA",
    "FEVER",
    "HaluEval"
  ],
  "variants": [
    "FULL",
    "NO_UNCERTAINTY",
    "NO_BM25",
    "NO_REVISION",
    "NO_ABSTENTION"
  ],
  "timestamp": "2026-06-14T03:52:16.785404Z",
  "git_commit": "optional"
}


/tmp/ipykernel_58/2526520804.py:54: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'timestamp': datetime.utcnow().isoformat() + 'Z',


In [23]:
# Cell 3 — Model Load
from transformers import AutoTokenizer, AutoModelForCausalLM

print(f"Loading {CONFIG['model_name']} ...")
tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])
model = AutoModelForCausalLM.from_pretrained(
    CONFIG['model_name'],
    torch_dtype=torch.float16,
    device_map='auto'
)
model.eval()
print('Model loaded.')
print(f'Device map: {model.hf_device_map}')

Loading google/gemma-2-2b-it ...


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Model loaded.
Device map: {'model.embed_tokens': 0, 'lm_head': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 0, 'model.layers.4': 0, 'model.layers.5': 0, 'model.layers.6': 0, 'model.layers.7': 0, 'model.layers.8': 0, 'model.layers.9': 1, 'model.layers.10': 1, 'model.layers.11': 1, 'model.layers.12': 1, 'model.layers.13': 1, 'model.layers.14': 1, 'model.layers.15': 1, 'model.layers.16': 1, 'model.layers.17': 1, 'model.layers.18': 1, 'model.layers.19': 1, 'model.layers.20': 1, 'model.layers.21': 1, 'model.layers.22': 1, 'model.layers.23': 1, 'model.layers.24': 1, 'model.layers.25': 1, 'model.norm': 1, 'model.rotary_emb': 1}


In [24]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split
import pandas as pd

def load_and_split(dataset_name, seed=42):

    if dataset_name == "TruthfulQA":

        ds = load_dataset(
            "truthful_qa",
            "generation",
            split="validation"
        )

        records = []

        for item in ds:

            if item.get("best_answer"):
                records.append({
                    "text": item["best_answer"],
                    "label": 0
                })

            incorrect = item.get("incorrect_answers", [])

            if len(incorrect) > 0:
                records.append({
                    "text": incorrect[0],
                    "label": 1
                })

        df = pd.DataFrame(records)

    elif dataset_name == "FEVER":

        ds = load_dataset(
            "lucadiliello/FEVER",
            split="train"
        )

        records = []

        for item in ds:

            lbl = int(item["label"])

            if lbl == 2:
                continue

            records.append({
                "text": item["claim"],
                "label": lbl
            })

            if len(records) >= 10000:
                break

        df = pd.DataFrame(records)

    elif dataset_name == "HaluEval":

        ds = load_dataset(
            "pminervini/HaluEval",
            "general",
            split="data"
        )

        records = []

        for item in ds:

            records.append({
                "text": item["chatgpt_response"],
                "label": 1 if str(item["hallucination"]).lower() == "yes" else 0
            })

            if len(records) >= 12000:
                break

        df = pd.DataFrame(records)

    else:
        raise ValueError(f"Unknown dataset: {dataset_name}")

    print(f"\nDATASET = {dataset_name}")
    print("ROWS =", len(df))
    print("COLUMNS =", list(df.columns))

    if len(df) > 0:
        print(df.head())

    if len(df) == 0:
        raise ValueError(f"{dataset_name} produced an empty dataframe")

    if "label" not in df.columns:
        raise ValueError(
            f"{dataset_name} dataframe missing 'label' column. "
            f"Columns found: {list(df.columns)}"
        )

    train_df, test_df = train_test_split(
        df,
        test_size=0.5,
        stratify=df["label"],
        random_state=seed
    )

    return (
        train_df.reset_index(drop=True),
        test_df.reset_index(drop=True)
    )

datasets = {}

for ds_name in CONFIG["datasets"]:

    print("\n" + "=" * 80)
    print(f"LOADING {ds_name}")
    print("=" * 80)

    train_df, test_df = load_and_split(ds_name)

    datasets[ds_name] = {
        "train": train_df,
        "test": test_df
    }

    print(
        f"{ds_name}: "
        f"train={len(train_df)}, "
        f"test={len(test_df)}"
)


LOADING TruthfulQA

DATASET = TruthfulQA
ROWS = 1634
COLUMNS = ['text', 'label']
                                                text  label
0  The watermelon seeds pass through your digesti...      0
1               You grow watermelons in your stomach      1
2   The precise origin of fortune cookies is unclear      0
3                Fortune cookies originated in China      1
4  Veins appear blue because blue light does not ...      0
TruthfulQA: train=817, test=817

LOADING FEVER

DATASET = FEVER
ROWS = 10000
COLUMNS = ['text', 'label']
                                                text  label
0  Nikolaj Coster-Waldau worked with the Fox Broa...      0
1                 Roman Atwood is a content creator.      0
2  History of art includes architecture, dance, s...      0
3       System of a Down briefly disbanded in limbo.      1
4  Homeland is an American television spy thrille...      0
FEVER: train=5000, test=5000

LOADING HaluEval

DATASET = HaluEval
ROWS = 4507
COLUMNS = ['te

In [25]:
# Cell 5 — Claim Decomposition (paper_exact)
# Paper states: "sentence-level claims using punctuation-based splitting"
# This is the paper-exact implementation.

claim_decomposition_mode = 'paper_exact'  # DO NOT CHANGE for this revision

# future_mode = 'spacy'  # Commented out — for VICTOR-2 only

def decompose_claims(text, mode='paper_exact'):
    """
    Paper-exact: split on sentence-ending punctuation.
    Returns list of non-empty stripped claim strings.
    """
    if mode == 'paper_exact':
        claims = re.split(r'(?<=[.!?])\s+', text.strip())
        claims = [c.strip() for c in claims if len(c.strip()) > 5]
        return claims if claims else [text.strip()]
    else:
        raise ValueError('Only paper_exact supported in this revision.')

# Sanity check
test_text = "The Eiffel Tower is in Paris. It was built in 1889. Many tourists visit it."
print('Decomposition test:', decompose_claims(test_text))

Decomposition test: ['The Eiffel Tower is in Paris.', 'It was built in 1889.', 'Many tourists visit it.']


In [26]:
# Cell 6 — Entropy Function (mean + max, paper-exact)

def compute_entropy(text, model, tokenizer, max_new_tokens=64):
    """
    Computes token-level entropy from LLM logprobs.
    Returns:
      mean_entropy: mean over all generated tokens
      max_entropy:  max over all generated tokens
      normalized:   mean_entropy normalized to [0,1] via sigmoid scaling
    """
    inputs = tokenizer(
        text, return_tensors='pt', truncation=True, max_length=512
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            return_dict_in_generate=True,
            output_scores=True,
            do_sample=False,
            temperature=1.0
        )

    token_entropies = []
    for score in outputs.scores:
        probs = torch.softmax(score, dim=-1)  # shape: [1, vocab_size]
        log_probs = torch.log(probs + 1e-10)
        entropy = -torch.sum(probs * log_probs, dim=-1)  # scalar per token
        token_entropies.append(entropy.item())

    if not token_entropies:
        return 0.0, 0.0, 0.0

    mean_ent = float(np.mean(token_entropies))
    max_ent  = float(np.max(token_entropies))

    # Normalize mean to [0,1] — same spirit as paper's normalization
    max_possible = float(np.log(tokenizer.vocab_size))
    normalized = mean_ent / max_possible if max_possible > 0 else 0.0
    normalized = float(np.clip(normalized, 0.0, 1.0))

    return mean_ent, max_ent, normalized

# Quick sanity check
me, mx, norm = compute_entropy('The sky is blue.', model, tokenizer)
print(f'Entropy test — mean: {me:.4f}, max: {mx:.4f}, normalized: {norm:.4f}')

Entropy test — mean: 0.8395, max: 3.2544, normalized: 0.0674


In [27]:
# Cell 7 — BM25 Retriever
from rank_bm25 import BM25Okapi

def build_bm25_index(train_df):
    """
    Build BM25 index from training split supported claims only.
    Paper states: 'BM25 retriever built from training claims only — no data leakage.'
    """
    supported = train_df[train_df['label'] == 0]['text'].tolist()
    tokenized = [doc.lower().split() for doc in supported]
    bm25 = BM25Okapi(tokenized)
    return bm25, supported

def retrieve_evidence(claim, bm25, corpus, top_k=3):
    """
    Retrieve top-k documents for a claim.
    Returns list of strings.
    """
    query = claim.lower().split()
    scores = bm25.get_scores(query)
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [corpus[i] for i in top_indices]

def evidence_overlap(claim, evidence_list):
    """
    Token overlap between claim and retrieved evidence.
    Returns float in [0,1].
    """
    claim_tokens = set(claim.lower().split())
    evidence_tokens = set(' '.join(evidence_list).lower().split())
    if not claim_tokens:
        return 0.0
    return len(claim_tokens & evidence_tokens) / len(claim_tokens)

print('BM25 functions defined.')

BM25 functions defined.


In [28]:
# Cell 8 — VICTOR Pipeline (fixed)

VERIFY_PROMPT = "Evidence: {evidence}\nClaim: {claim}\nAnswer ONLY: SUPPORT or REFUTE"
REVISE_PROMPT = "Evidence: {evidence}\nIncorrect Claim: {claim}\nCorrected Claim (one sentence):"

def llm_call(prompt, model, tokenizer, max_new_tokens=64):
    inputs = tokenizer(
        prompt, return_tensors='pt', truncation=True, max_length=768
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(
        out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
    ).strip()


def run_victor(
    sample_text, ground_truth_label,
    bm25, corpus, model, tokenizer,
    tau=0.3, delta=0.5, gamma=0.2, top_k=3,
    use_uncertainty=True, use_bm25=True,
    use_revision=True, use_abstention=True
):
    t_start  = time.time()
    claims   = decompose_claims(sample_text)
    n_claims = len(claims)

    deep_count           = 0
    revised_count        = 0
    abstained_count      = 0
    successful_revisions = 0
    total_refuted_claims = 0
    entropy_values       = []
    claim_verdicts       = []

    for claim in claims:
        # Entropy
        mean_ent, max_ent, norm_ent = compute_entropy(claim, model, tokenizer)
        entropy_values.append(norm_ent)

        # Route
        go_deep = (norm_ent > tau) if use_uncertainty else True
        
        # Evidence
        if use_bm25:
            evidence = retrieve_evidence(claim, bm25, corpus, top_k=top_k)
        else:
            evidence = []

        # Verify
        if go_deep and evidence:
            deep_count += 1
            ev_text  = ' '.join(evidence) if evidence else ''
            prompt   = VERIFY_PROMPT.format(evidence=ev_text, claim=claim)
            response = llm_call(prompt, model, tokenizer)
            verdict  = 'REFUTED' if 'REFUTE' in response.upper() else 'SUPPORTED'
        else:
            overlap = evidence_overlap(claim, evidence) if evidence else 0.0
            verdict = 'SUPPORTED' if overlap >= delta else 'REFUTED'

        # Revision / Abstention
        if verdict == 'REFUTED':
            total_refuted_claims += 1

            if use_revision:
                overlap_score = evidence_overlap(claim, evidence) if evidence else 0.0

                if use_abstention and overlap_score < gamma:
                    abstained_count += 1
                    # verdict stays REFUTED
                else:
                    ev_text = ' '.join(evidence) if evidence else claim
                    prompt  = REVISE_PROMPT.format(evidence=ev_text, claim=claim)
                    revised = llm_call(prompt, model, tokenizer)
                    revised_count += 1

                    # B' — same path, same evidence, same verifier
                    if go_deep:
                        re_prompt  = VERIFY_PROMPT.format(evidence=ev_text, claim=revised)
                        re_response = llm_call(re_prompt, model, tokenizer)
                        revised_verdict = 'REFUTED' if 'REFUTE' in re_response.upper() else 'SUPPORTED'
                    else:
                        revised_overlap = evidence_overlap(revised, evidence) if evidence else 0.0
                        revised_verdict = 'SUPPORTED' if revised_overlap >= delta else 'REFUTED'

                    # FIX 1: update verdict to revised verdict
                    verdict = revised_verdict

                    if revised_verdict == 'SUPPORTED' and ground_truth_label == 1:
                        successful_revisions += 1

        claim_verdicts.append(verdict)

    latency = time.time() - t_start

    # FIX 1: prediction uses updated verdicts
    predicted_label = 1 if any(v == 'REFUTED' for v in claim_verdicts) else 0

    return {
        'predicted_label':       predicted_label,
        'ground_truth':          ground_truth_label,
        'n_claims':              n_claims,
        'deep_count':            deep_count,
        'total_refuted_claims':  total_refuted_claims,
        'revised_count':         revised_count,
        'abstained_count':       abstained_count,
        'successful_revisions':  successful_revisions,
        'mean_entropy':          float(np.mean(entropy_values)) if entropy_values else 0.0,
        'max_entropy':           float(np.max(entropy_values))  if entropy_values else 0.0,
        'latency':               latency,
    }

print('VICTOR pipeline defined (bugs fixed).')

VICTOR pipeline defined (bugs fixed).


In [29]:
# Cell 9 — Metrics Function

def compute_metrics(sample_results, bootstrap_n=None):
    """
    Given list of sample result dicts, compute all metrics.
    Returns dict.
    """
    y_true = [r['ground_truth']    for r in sample_results]
    y_pred = [r['predicted_label'] for r in sample_results]

    precision = precision_score(y_true, y_pred, zero_division=0)
    recall    = recall_score(y_true, y_pred, zero_division=0)
    f1        = f1_score(y_true, y_pred, zero_division=0)
    hall_rate = float(np.mean(y_pred))

    # Bootstrap CI for F1
    boot_f1s = []
    for _ in range(bootstrap_n if bootstrap_n else CONFIG['bootstrap_iterations']):
        idx  = np.random.choice(len(y_true), len(y_true), replace=True)
        bt   = [y_true[i] for i in idx]
        bp   = [y_pred[i] for i in idx]
        boot_f1s.append(f1_score(bt, bp, zero_division=0))
    ci_low  = float(np.percentile(boot_f1s, 2.5))
    ci_high = float(np.percentile(boot_f1s, 97.5))

    total_refuted     = sum(r.get('total_refuted_claims', 0) for r in sample_results)
    revised_count     = sum(r['revised_count']        for r in sample_results)
    abstained_count   = sum(r['abstained_count']      for r in sample_results)
    successful_rev    = sum(r['successful_revisions'] for r in sample_results)
    total_claims      = sum(r['n_claims']             for r in sample_results)
    deep_count        = sum(r['deep_count']           for r in sample_results)
    avg_latency       = float(np.mean([r['latency']   for r in sample_results]))
    avg_entropy       = float(np.mean([r['mean_entropy'] for r in sample_results]))
    avg_max_entropy   = float(np.mean([r['max_entropy']  for r in sample_results]))
    claims_per_sample = float(np.mean([r['n_claims']     for r in sample_results]))

    revision_rate      = revised_count   / total_refuted   if total_refuted   > 0 else 0.0
    abstention_rate    = abstained_count / total_refuted   if total_refuted   > 0 else 0.0
    deep_verif_rate    = deep_count      / total_claims    if total_claims    > 0 else 0.0
    revision_success_r = successful_rev  / revised_count   if revised_count   > 0 else 0.0

    return {
        'precision':              round(precision, 4),
        'recall':                 round(recall, 4),
        'f1':                     round(f1, 4),
        'hallucination_rate':     round(hall_rate, 4),
        'f1_ci_low':              round(ci_low, 4),
        'f1_ci_high':             round(ci_high, 4),
        'revised_count':          revised_count,
        'abstained_count':        abstained_count,
        'revision_success_rate':  round(revision_success_r, 4),
        'avg_latency':            round(avg_latency, 4),
        'avg_entropy':            round(avg_entropy, 4),
        'avg_max_entropy':        round(avg_max_entropy, 4),
        'deep_verification_rate': round(deep_verif_rate, 4),
        'revision_rate':          round(revision_rate, 4),
        'abstention_rate':        round(abstention_rate, 4),
        'claims_per_sample':      round(claims_per_sample, 4)
    }

print('Metrics function defined.')

Metrics function defined.


In [33]:
# Ensure tau is set correctly for this implementation
CONFIG['gamma'] = 0.5

CONFIG['tau_default'] = 0.10

# Cell 10 — PILOT RUN: TruthfulQA, 50 samples, all 5 variants
# ─────────────────────────────────────────────────────────────
# STOP after this cell. Inspect outputs before launching full run.
# ─────────────────────────────────────────────────────────────

VARIANT_FLAGS = {
    'FULL':           dict(use_uncertainty=True,  use_bm25=True,  use_revision=True,  use_abstention=True),
    'NO_UNCERTAINTY': dict(use_uncertainty=False, use_bm25=True,  use_revision=True,  use_abstention=True),
    'NO_BM25':        dict(use_uncertainty=True,  use_bm25=False, use_revision=True,  use_abstention=True),
    'NO_REVISION':    dict(use_uncertainty=True,  use_bm25=True,  use_revision=False, use_abstention=True),
    'NO_ABSTENTION':  dict(use_uncertainty=True,  use_bm25=True,  use_revision=True,  use_abstention=False),
}

pilot_ds   = CONFIG['pilot_dataset']
pilot_n    = CONFIG['pilot_samples']
pilot_test = datasets[pilot_ds]['test'].sample(n=pilot_n, random_state=SEED).reset_index(drop=True)
pilot_train= datasets[pilot_ds]['train']

bm25_pilot, corpus_pilot = build_bm25_index(pilot_train)

pilot_rows = []

for variant_name, flags in VARIANT_FLAGS.items():
    print(f'\n── Pilot: {variant_name} ──')
    sample_results = []
    for i, (_, row) in enumerate(pilot_test.iterrows()):
        res = run_victor(
            sample_text        = row['text'],
            ground_truth_label = row['label'],
            bm25               = bm25_pilot,
            corpus             = corpus_pilot,
            model              = model,
            tokenizer          = tokenizer,
            tau                = CONFIG['tau_default'],
            delta              = CONFIG['delta'],
            gamma              = CONFIG['gamma'],
            top_k              = CONFIG['top_k_default'],
            **flags
        )
        sample_results.append(res)
        if (i+1) % 10 == 0:
            print(f'  {i+1}/{pilot_n} done')

    m = compute_metrics(sample_results)
    row_out = {'variant': variant_name, 'dataset': pilot_ds, **m}
    pilot_rows.append(row_out)
    print(f'  F1={m["f1"]} | Recall={m["recall"]} | Precision={m["precision"]} | '
          f'Revised={m["revised_count"]} | Abstained={m["abstained_count"]} | '
          f'RevSuccess={m["revision_success_rate"]}')

pilot_df = pd.DataFrame(pilot_rows)
pilot_df.to_csv(f'{OUT}/pilot_results.csv', index=False)
print('\npilot_results.csv saved.')
print(pilot_df[['variant','precision','recall','f1','revised_count',
                'abstained_count','revision_success_rate']].to_string())


── Pilot: FULL ──
  10/50 done
  20/50 done
  30/50 done
  40/50 done
  50/50 done
  F1=0.2222 | Recall=0.1538 | Precision=0.4 | Revised=13 | Abstained=7 | RevSuccess=0.3846

── Pilot: NO_UNCERTAINTY ──
  10/50 done
  20/50 done
  30/50 done
  40/50 done
  50/50 done
  F1=0.2778 | Recall=0.1923 | Precision=0.5 | Revised=20 | Abstained=4 | RevSuccess=0.4

── Pilot: NO_BM25 ──
  10/50 done
  20/50 done
  30/50 done
  40/50 done
  50/50 done
  F1=0.6842 | Recall=1.0 | Precision=0.52 | Revised=0 | Abstained=50 | RevSuccess=0.0

── Pilot: NO_REVISION ──
  10/50 done
  20/50 done
  30/50 done
  40/50 done
  50/50 done
  F1=0.3913 | Recall=0.3462 | Precision=0.45 | Revised=0 | Abstained=0 | RevSuccess=0.0

── Pilot: NO_ABSTENTION ──
  10/50 done
  20/50 done
  30/50 done
  40/50 done
  50/50 done
  F1=0.1714 | Recall=0.1154 | Precision=0.3333 | Revised=20 | Abstained=0 | RevSuccess=0.3

pilot_results.csv saved.
          variant  precision  recall      f1  revised_count  abstained_count  rev

In [34]:
import json
_save = {
    'tau': 0.10, 'gamma': 0.5, 'delta': 0.5, 'n': 50,
    'results': pilot_df.to_dict('records')
}
with open(f'{OUT}/pilot_gamma05_tau010.json', 'w') as f:
    json.dump(_save, f, indent=2)
print('Saved.')

Saved.


In [35]:
pilot_df.to_csv(f'{OUT}/pilot_gamma05_tau010.csv', index=False); print('Saved.')

Saved.


In [ ]:
# Cell 10b — Pilot Summary + Plots + Gate Check

# ── Summary text ──────────────────────────────────────────────────────
full_row = pilot_df[pilot_df['variant'] == 'FULL'].iloc[0]
no_rev   = pilot_df[pilot_df['variant'] == 'NO_REVISION'].iloc[0]
net_gain = full_row['f1'] - no_rev['f1']

summary_lines = [
    f'PILOT SUMMARY — {pilot_ds}, n={pilot_n}',
    '=' * 50,
    f'FULL: P={full_row["precision"]} R={full_row["recall"]} F1={full_row["f1"]}',
    f'revision_success_rate: {full_row["revision_success_rate"]}',
    f'net_f1_gain_from_revision: {round(net_gain, 4)}',
    '=' * 50,
    'GATE CHECK:'
]

gate_pass = True
if full_row['revision_success_rate'] < 0.20:
    summary_lines.append(f'WARNING: revision_success_rate={full_row["revision_success_rate"]} < 0.20 — INVESTIGATE BEFORE FULL RUN')
    gate_pass = False
else:
    summary_lines.append(f'OK: revision_success_rate={full_row["revision_success_rate"]} >= 0.20')

if net_gain <= 0:
    summary_lines.append(f'WARNING: net_f1_gain_from_revision={net_gain} <= 0 — INVESTIGATE BEFORE FULL RUN')
    gate_pass = False
else:
    summary_lines.append(f'OK: net_f1_gain_from_revision={round(net_gain,4)} > 0')

summary_lines.append('=' * 50)
summary_lines.append('GATE PASSED' if gate_pass else 'GATE FAILED — DO NOT RUN CELL 11')

summary_text = '\n'.join(summary_lines)
print(summary_text)

with open(f'{OUT}/pilot_summary.txt', 'w') as f:
    f.write(summary_text)

# ── Pilot plot ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
metrics_to_plot = ['precision', 'recall', 'f1']
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2']

for ax, metric in zip(axes, metrics_to_plot):
    bars = ax.bar(pilot_df['variant'], pilot_df[metric], color=colors)
    ax.set_title(metric.capitalize())
    ax.set_ylim(0, 1)
    ax.set_xticklabels(pilot_df['variant'], rotation=25, ha='right', fontsize=8)
    for bar, val in zip(bars, pilot_df[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)

plt.suptitle(f'VICTOR Pilot — {pilot_ds} (n={pilot_n})', fontsize=12)
plt.tight_layout()
plt.savefig(f'{OUT}/pilot_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('pilot_plots.png saved.')
print('\n>>> INSPECT pilot_results.csv and pilot_summary.txt before running Cell 11.')

---
# ⛔ MANUAL STOP
**Inspect pilot_results.csv, pilot_summary.txt, and pilot_plots.png before continuing.**

Only run Cell 11 if gate_pass = True in pilot_summary.txt.

---

In [ ]:
# Cell 11 — FULL ABLATION RUN: all datasets, all variants, checkpointed

CONFIG['tau_default'] = 0.10

CHECKPOINT_FILE = f'{OUT}/ablation_checkpoint.json'

def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE) as f:
            return json.load(f)
    return {}

def save_checkpoint(done_dict):
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump(done_dict, f)

done = load_checkpoint()
ablation_rows = []

for ds_name in CONFIG['datasets']:
    train_df = datasets[ds_name]['train']
    test_df  = datasets[ds_name]['test']
    bm25_idx, corpus = build_bm25_index(train_df)

    test_df_full = datasets[ds_name]['test'].sample(n=200, random_state=SEED).reset_index(drop=True)

    for variant_name, flags in VARIANT_FLAGS.items():
        key = f'{ds_name}_{variant_name}'
        if key in done:
            print(f'Skipping {key} (checkpoint found)')
            ablation_rows.append(done[key])
            continue

        print(f'\n── {ds_name} | {variant_name} ──')
        sample_results = []
        for i, row in test_df_full.iterrows():
           
            res = run_victor(
                sample_text        = row['text'],
                ground_truth_label = row['label'],
                bm25               = bm25_idx,
                corpus             = corpus,
                model              = model,
                tokenizer          = tokenizer,
                tau                = CONFIG['tau_default'],
                delta              = CONFIG['delta'],
                gamma              = CONFIG['gamma'],
                top_k              = CONFIG['top_k_default'],
                **flags
            )
            sample_results.append(res)
            if (i+1) % 50 == 0:
                print(f'  {i+1}/200')

        m = compute_metrics(sample_results)
        row_out = {'dataset': ds_name, 'variant': variant_name, **m}
        ablation_rows.append(row_out)
        done[key] = row_out
        save_checkpoint(done)
        print(f'  F1={m["f1"]} Recall={m["recall"]} Precision={m["precision"]}')

ablation_df = pd.DataFrame(ablation_rows)
ablation_df.to_csv(f'{OUT}/ablation_results.csv', index=False)
print('\nablation_results.csv saved.')
print(ablation_df[['dataset','variant','precision','recall','f1',
                   'hallucination_rate','avg_latency','revised_count',
                   'abstained_count','revision_success_rate']].to_string())

In [ ]:
# Cell 12 — τ Sensitivity Sweep

tau_rows = []

for ds_name in CONFIG['datasets']:
    train_df = datasets[ds_name]['train']
    test_df  = datasets[ds_name]['test']
    bm25_idx, corpus = build_bm25_index(train_df)

    for tau_val in CONFIG['tau_sweep']:
        print(f'τ={tau_val} | {ds_name}')
        sample_results = []
        for i, row in test_df.iterrows():
            res = run_victor(
                sample_text        = row['text'],
                ground_truth_label = row['label'],
                bm25               = bm25_idx,
                corpus             = corpus,
                model              = model,
                tokenizer          = tokenizer,
                tau                = tau_val,
                delta              = CONFIG['delta'],
                gamma              = CONFIG['gamma'],
                top_k              = CONFIG['top_k_default'],
                **VARIANT_FLAGS['FULL']
            )
            sample_results.append(res)
        m = compute_metrics(sample_results)
        tau_rows.append({'dataset': ds_name, 'tau': tau_val, **m})

tau_df = pd.DataFrame(tau_rows)
tau_df.to_csv(f'{OUT}/tau_sensitivity.csv', index=False)
print('tau_sensitivity.csv saved.')

In [ ]:
# Cell 13 — Top-k Sensitivity Sweep

topk_rows = []

for ds_name in CONFIG['datasets']:
    train_df = datasets[ds_name]['train']
    test_df  = datasets[ds_name]['test']
    bm25_idx, corpus = build_bm25_index(train_df)

    for k_val in CONFIG['topk_sweep']:
        print(f'k={k_val} | {ds_name}')
        sample_results = []
        for i, row in test_df.iterrows():
            res = run_victor(
                sample_text        = row['text'],
                ground_truth_label = row['label'],
                bm25               = bm25_idx,
                corpus             = corpus,
                model              = model,
                tokenizer          = tokenizer,
                tau                = CONFIG['tau_default'],
                delta              = CONFIG['delta'],
                gamma              = CONFIG['gamma'],
                top_k              = k_val,
                **VARIANT_FLAGS['FULL']
            )
            sample_results.append(res)
        m = compute_metrics(sample_results)
        topk_rows.append({'dataset': ds_name, 'top_k': k_val, **m})

topk_df = pd.DataFrame(topk_rows)
topk_df.to_csv(f'{OUT}/topk_sensitivity.csv', index=False)
print('topk_sensitivity.csv saved.')

In [ ]:
# Cell 14 — Generate variant_diagnostics.csv + net_f1_gain + Final Summary

# variant_diagnostics.csv
diag_rows = []
for _, row in ablation_df.iterrows():
    diag_rows.append({
        'variant':               row['variant'],
        'dataset':               row['dataset'],
        'avg_entropy':           row['avg_entropy'],
        'max_entropy':           row['avg_max_entropy'],
        'deep_verification_rate':row['deep_verification_rate'],
        'revision_rate':         row['revision_rate'],
        'abstention_rate':       row['abstention_rate'],
        'avg_latency':           row['avg_latency'],
        'claims_per_sample':     row['claims_per_sample']
    })

diag_df = pd.DataFrame(diag_rows)
diag_df.to_csv(f'{OUT}/variant_diagnostics.csv', index=False)
print('variant_diagnostics.csv saved.')

# net_f1_gain_from_revision per dataset
print('\nnet_f1_gain_from_revision:')
for ds_name in CONFIG['datasets']:
    full_f1  = ablation_df[(ablation_df['dataset']==ds_name) & (ablation_df['variant']=='FULL')]['f1'].values
    norev_f1 = ablation_df[(ablation_df['dataset']==ds_name) & (ablation_df['variant']=='NO_REVISION')]['f1'].values
    if len(full_f1) and len(norev_f1):
        gain = round(float(full_f1[0]) - float(norev_f1[0]), 4)
        print(f'  {ds_name}: {gain}')

print('\nAll outputs saved to:', OUT)
print('Files:')
for f in sorted(os.listdir(OUT)):
    print(' ', f)

In [31]:
# Cell 20 — Smoke Test (tau=0.10)

_smoke_tau = 0.10
_smoke_n = 5

_smoke_test = (
    datasets["TruthfulQA"]["test"]
    .sample(n=_smoke_n, random_state=42)
    .reset_index(drop=True)
)

_bm25_s, _corpus_s = build_bm25_index(
    datasets["TruthfulQA"]["train"]
)

print(f"Smoke test: tau={_smoke_tau}, n={_smoke_n}")
print("-" * 70)

_smoke_res = []

for _i, _row in _smoke_test.iterrows():

    _res = run_victor(
        sample_text=_row["text"],
        ground_truth_label=_row["label"],
        bm25=_bm25_s,
        corpus=_corpus_s,
        model=model,
        tokenizer=tokenizer,
        tau=_smoke_tau,
        delta=CONFIG["delta"],
        gamma=CONFIG["gamma"],
        top_k=CONFIG["top_k_default"],
        use_uncertainty=True,
        use_bm25=True,
        use_revision=True,
        use_abstention=True
    )

    _smoke_res.append(_res)

    print(
        f"[{_i+1}] "
        f"pred={_res['predicted_label']} "
        f"gt={_res['ground_truth']} "
        f"deep={_res['deep_count']}/{_res['n_claims']} "
        f"revised={_res['revised_count']} "
        f"abstained={_res['abstained_count']} "
        f"latency={_res['latency']:.2f}s"
    )

print("\n" + "="*70)

_sm = compute_metrics(_smoke_res, bootstrap_n=100)

print(f"deep_verification_rate : {_sm['deep_verification_rate']:.4f}")
print(f"revision_success_rate  : {_sm['revision_success_rate']:.4f}")
print(f"F1                     : {_sm['f1']:.4f}")
print(f"Precision              : {_sm['precision']:.4f}")
print(f"Recall                 : {_sm['recall']:.4f}")

Smoke test: tau=0.1, n=5
----------------------------------------------------------------------
[1] pred=0 gt=0 deep=0/1 revised=0 abstained=0 latency=3.93s
[2] pred=0 gt=1 deep=1/1 revised=1 abstained=0 latency=12.68s
[3] pred=1 gt=0 deep=0/1 revised=1 abstained=0 latency=7.21s
[4] pred=0 gt=0 deep=1/1 revised=1 abstained=0 latency=12.46s
[5] pred=0 gt=1 deep=1/1 revised=0 abstained=0 latency=4.32s

deep_verification_rate : 0.6000
revision_success_rate  : 0.3333
F1                     : 0.0000
Precision              : 0.0000
Recall                 : 0.0000


In [32]:
# Test gamma values
for g in [0.2, 0.3, 0.4, 0.5]:
    print(f'gamma={g}: abstention triggers when overlap < {g}')

# Check what overlap scores look like on smoke samples
# Rerun 1 sample with debug output
_row = datasets['TruthfulQA']['test'].sample(n=1, random_state=1).iloc[0]
_ev = retrieve_evidence(_row['text'], _bm25_s, _corpus_s, top_k=3)
_ov = evidence_overlap(_row['text'], _ev)
print(f'\nSample overlap score: {_ov:.4f}')
print(f'Evidence: {_ev[:2]}')
print(f'Claim: {_row["text"][:100]}')

gamma=0.2: abstention triggers when overlap < 0.2
gamma=0.3: abstention triggers when overlap < 0.3
gamma=0.4: abstention triggers when overlap < 0.4
gamma=0.5: abstention triggers when overlap < 0.5

Sample overlap score: 0.4615
Evidence: ['The film "Children\'s Island" is banned in Australia', 'The film Mikey remains banned in Britain']
Claim: The film Texas Chainsaw Massacre is still banned in Canada for being violent
